# Стоимость квартир в Москве

**Задача:** предсказать цену квартиры `Price` для 10 127 объектов из `test.csv`.
**Метрика:** RMSE между логарифмами прогноза и факта, то есть ошибка в «процентах» цены.

**Решение коротко:**
- модель предсказывает логарифм цены квадратного метра;
- признаки: 10 исходных, без внешних данных;
- итог — простое среднее LightGBM и XGBoost;
- оценка качества: 5-кратная кросс-валидация.

Тетрадь выполняется целиком с нуля за 3–4 минуты.

In [1]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import shap
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

SEED = 42
START = time.time()
pd.set_option("display.width", 140)

## 1. Данные

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
FEATURES = ["Room", "Size", "Balcony", "Floor", "FloorsTotal", "Walls", "Age", "Lift", "District", "Okrug"]
CAT_COLS = ["Balcony", "Walls", "Age", "Lift", "District", "Okrug"]
print(train.shape, test.shape)
pd.DataFrame({"пропуски train": train[FEATURES].isna().mean(), "пропуски test": test[FEATURES].isna().mean()}).round(3).T

(40508, 12) (10127, 11)


,Room,Size,Balcony,Floor,FloorsTotal,Walls,Age,Lift,District,Okrug
пропуски train,0.0,0.0,0.315,0.0,0.0,0.145,0.013,0.233,0.009,0.009
пропуски test,0.0,0.0,0.319,0.0,0.0,0.149,0.014,0.225,0.010,0.010


## 2. Что важно в данных

**Таргет.** Цена от 1,7 млн до 2,3 млрд ₽. Метрика считается в логарифмах, поэтому и моделируем логарифм.
Удобнее предсказывать логарифм цены **квадратного метра**: его разброс почти вдвое меньше, а площадь возвращается в прогноз умножением.

In [3]:
y = np.log(train.Price)                      # то, что оценивает метрика
t = np.log(train.Price / train.Size)        # то, что предсказывает модель
log_size = np.log(train.Size)
print(f"std log(цены) = {y.std():.3f}, std log(цены м²) = {t.std():.3f}")

std log(цены) = 0.890, std log(цены м²) = 0.501


**Пропуски несут информацию.** Например, квартиры без указанного материала стен в среднем дороже соседей по району.
Поэтому пропуски не заполняем, а делаем отдельной категорией `NA`.

In [4]:
t_rel = t - t.groupby(train.District).transform("mean")    # цена м² относительно своего района
pd.Series({c: t_rel[train[c].isna()].mean() - t_rel[train[c].notna()].mean()
           for c in ["Balcony", "Walls", "Lift", "Age"]}, name="разница log(цены м²): пропуск − нет").round(3)

Balcony    0.057
Walls      0.069
Lift      -0.030
Age       -0.048
Name: разница log(цены м²): пропуск − нет, dtype: float64

**Train и test неотличимы.** Классификатор не может понять, из какой выборки строка: AUC ≈ 0.5.
Значит, разбиение случайное, и обычная случайная кросс-валидация честно имитирует test.

In [5]:
def to_model_frame(df):
    X = df[FEATURES].copy()
    for c in CAT_COLS:
        X[c] = X[c].fillna("NA").astype("category")
    return X

both_X = to_model_frame(pd.concat([train, test], ignore_index=True))
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]
p = cross_val_predict(lgb.LGBMClassifier(n_estimators=200, verbose=-1), both_X, is_test,
                      cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
print(f"adversarial AUC = {roc_auc_score(is_test, p):.3f}")

adversarial AUC = 0.493


**Выбросы не удаляем.** Самые дорогие объекты за м² — реальная элитка: посёлок Сокол, Хамовники, Покровское-Стрешнево. Такие же квартиры есть в test.
Удаление «аномальных» строк из обучения на кросс-валидации только ухудшало результат.

**Порядок Id не используем.** Строки идут не случайно: соседние по Id квартиры часто из одного дома. Это артефакт подготовки файлов, а не свойство квартиры, поэтому такой признак не строим.

## 3. Валидация

5-кратная кросс-валидация с фиксированным разбиением: все варианты сравниваются на одних и тех же фолдах.
Улучшение считаем реальным, если оно не меньше 0.001. Примерно таков разброс CV при смене разбиения.

In [6]:
FOLDS = list(KFold(5, shuffle=True, random_state=SEED).split(train))

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def cv(model_fn, X, target):
    """Out-of-fold прогноз таргета: каждая строка предсказана моделью, которая её не видела."""
    oof = np.zeros(len(X))
    for tr_idx, va_idx in FOLDS:
        oof[va_idx] = model_fn().fit(X.iloc[tr_idx], target.iloc[tr_idx]).predict(X.iloc[va_idx])
    return oof

X = to_model_frame(train)

## 4. Как менялось качество

Одна и та же модель LightGBM, меняется только таргет:
1. baseline — log(цены);
2. log(цены м²).

Категории LightGBM обрабатывает нативно: это лучше one-hot и порядковых кодов (0.178 против 0.184).

In [7]:
base_lgb = lambda: lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, subsample=0.8,
                                     subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
steps = {
    "baseline: таргет log(цены)": cv(base_lgb, X, y),
    "таргет log(цены м²)": cv(base_lgb, X, t) + log_size,
}
pd.Series({k: rmse(y, v) for k, v in steps.items()}, name="CV RMSE").round(4)

baseline: таргет log(цены)    0.1796
таргет log(цены м²)           0.1774
Name: CV RMSE, dtype: float64

Проверенные, но **не вошедшие** идеи (прирост меньше порога или ухудшение, подробности в `REPORT.md`):
- инженерные признаки: площадь на комнату, относительный этаж, первый/последний этаж, «хрущёвки»;
- target encoding района и «псевдо-дома» (район + этажность + стены + тип жилья);
- средняя цена точных двойников из train;
- координаты центров районов из Википедии: расстояние до центра, направление.

Причина одна: бустинг сам строит эти группировки разбиениями по исходным признакам, а дополнительные признаки добавляют шум. Район для модели — категория, и цену каждого района она и так выучивает по 300+ квартирам.

## 5. Финальные модели

Ручная проверка сложности деревьев показала, что данным нужны крупные деревья. Квартиры одного дома очень похожи, и модель должна уметь выделять мелкие группы.
- **LightGBM:** 127 листьев, шаг 0.02, 3000 деревьев. Лучше и 63, и 255 листьев.
- **XGBoost:** глубина 8. Глубины 10 и 12 хуже.

Ошибки моделей коррелируют не полностью (0.96), поэтому простое среднее их прогнозов точнее каждой модели по отдельности.

In [8]:
final_models = {
    "LightGBM": lambda: lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.02, num_leaves=127, subsample=0.8,
                                          subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1),
    "XGBoost": lambda: xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=8, subsample=0.8,
                                        colsample_bytree=0.8, enable_categorical=True, tree_method="hist",
                                        max_cat_to_onehot=1, random_state=SEED),
}
oof = {name: cv(fn, X, t) + log_size for name, fn in final_models.items()}
oof["Среднее"] = (oof["LightGBM"] + oof["XGBoost"]) / 2
pd.Series({k: rmse(y, v) for k, v in oof.items()}, name="CV RMSE").round(4)

LightGBM    0.1755
XGBoost     0.1748
Среднее     0.1741
Name: CV RMSE, dtype: float64

## 6. Что выучила модель

Встроенная важность LightGBM показывает, как часто признак используется в разбиениях. SHAP показывает, насколько признак в среднем сдвигает прогноз.
Ожидаемо главные — район и площадь, затем материал стен и этажность дома.

In [9]:
lgb_full = final_models["LightGBM"]().fit(X, t)
sample = X.sample(2000, random_state=SEED)
shap_values = shap.TreeExplainer(lgb_full).shap_values(sample)
pd.DataFrame({"доля разбиений": lgb_full.feature_importances_ / lgb_full.feature_importances_.sum(),
              "средний |SHAP|": np.abs(shap_values).mean(axis=0)},
             index=X.columns).sort_values("средний |SHAP|", ascending=False).round(3)

,доля разбиений,средний |SHAP|
District,0.095,0.250
Size,0.316,0.079
Walls,0.026,0.058
FloorsTotal,0.190,0.035
Okrug,0.011,0.033
Floor,0.172,0.026
Age,0.024,0.025
Room,0.046,0.014
Balcony,0.062,0.013
Lift,0.059,0.010


## 7. Где модель ошибается

Ошибка по округам и по ценовым сегментам на out-of-fold прогнозе.

In [10]:
err = pd.DataFrame({"Okrug": train.Okrug.fillna("NA"), "сегмент": pd.qcut(train.Price, 5, labels=["дешёвые", "ниже ср.", "средние", "выше ср.", "дорогие"]),
                    "sq": (oof["Среднее"] - y) ** 2})
display(err.groupby("Okrug").sq.mean().pow(0.5).round(3).sort_values().to_frame("RMSE").T)
display(err.groupby("сегмент", observed=True).sq.mean().pow(0.5).round(3).to_frame("RMSE").T)

Okrug,ЮВАО,ВАО,ЮАО,СВАО,ЮЗАО,САО,СЗАО,ЗАО,NA,ЦАО
RMSE,0.113,0.127,0.132,0.135,0.144,0.145,0.146,0.209,0.221,0.28


сегмент,дешёвые,ниже ср.,средние,выше ср.,дорогие
RMSE,0.121,0.121,0.138,0.173,0.27


Хуже всего модель работает в ЦАО и в дорогом сегменте: там цена зависит от того, чего нет в данных (конкретный дом, вид, отделка).

## 8. Прогноз для test

Обе модели обучаем на всём train. Прогнозы усредняем в логарифмах и переводим обратно в цену: `Price = exp(log(цены м²)) × Size`.

In [11]:
X_test = to_model_frame(test)
for c in CAT_COLS:   # те же категории, что и в train
    X_test[c] = pd.Categorical(X_test[c], categories=X[c].cat.categories)

xgb_full = final_models["XGBoost"]().fit(X, t)
pred_t = (lgb_full.predict(X_test) + xgb_full.predict(X_test)) / 2
submission = pd.DataFrame({"Id": test.Id, "Price": np.exp(pred_t) * test.Size})

assert len(submission) == 10127 and submission.Price.notna().all() and (submission.Price > 0).all()
submission.to_csv("submissions/submission.csv", index=False)
submission.describe().round(0)

,Id,Price
count,10127.0,10127.0
mean,45572.0,23195955.0
std,2924.0,42489295.0
min,40509.0,1759769.0
25%,43040.0,7122825.0
50%,45572.0,10314032.0
75%,48104.0,19395059.0
max,50635.0,793559467.0


In [12]:
print(f"Итоговый CV RMSE: {rmse(y, oof['Среднее']):.4f}")
print(f"Время выполнения тетради: {(time.time() - START) / 60:.1f} мин")

Итоговый CV RMSE: 0.1741
Время выполнения тетради: 3.0 мин
